**Importing Libraries**

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from datasets import Dataset

!pip install evaluate
import evaluate

from transformers import (
    DistilBertTokenizerFast,
    DistilBertForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
)

import torch

Reading Dataset

In [ ]:
df = pd.read_csv("MESC.csv")

# Filter missing values
df = df.dropna(subset=["Utterance", "Emotion"])
df.head()


,Utterance,Speaker,Emotion,Strategy,Dialogue_ID,Utterance_ID,Season,Episode,StartTime,EndTime
0,I told you.,Client,sadness,undefined,0,0,1,1,"00:00:54,097","00:00:55,113"
1,Told me what?,Therapist,neutral,Open question,0,1,1,1,"00:00:55,260","00:00:56,263"
2,That you'd be sorry you ever encouraged me to ...,Client,sadness,undefined,0,2,1,1,"00:00:56,651","00:00:59,719"
3,I'm not sorry at all.,Therapist,neutral,Communication Skills,0,3,1,1,"00:00:59,951","00:01:01,237"
4,"You didn't expect it to be like this, I bet.",Client,sadness,undefined,0,4,1,1,"00:01:03,404","00:01:05,206"


In [ ]:
le = LabelEncoder()
df["label"] = le.fit_transform(df["Emotion"])
num_labels = len(le.classes_)
print("Classes:", le.classes_)

Classes: ['anger' 'depression' 'disgust' 'fear' 'joy' 'neutral' 'sadness']


In [ ]:
train_df, val_df = train_test_split(
    df,
    test_size=0.1,
    random_state=42,
    stratify=df["label"]
)

train_ds = Dataset.from_pandas(train_df)
val_ds   = Dataset.from_pandas(val_df)

Tokenizer

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("roberta-base")

def tokenize(batch):
    return tokenizer(
        batch["Utterance"],
        truncation=True,
        max_length=128,
    )

train_ds = train_ds.map(tokenize, batched=True)
val_ds   = val_ds.map(tokenize, batched=True)

Map:   0%|          | 0/25885 [00:00<?, ? examples/s]

Map:   0%|          | 0/2877 [00:00<?, ? examples/s]

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    "roberta-base",
    num_labels=num_labels
)

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [ ]:
accuracy = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return accuracy.compute(predictions=preds, references=labels)

In [ ]:
import transformers
from transformers import TrainingArguments

print("Transformers version:", transformers.__version__)
print("TrainingArguments module:", TrainingArguments.__module__)

Transformers version: 4.57.2
TrainingArguments module: transformers.training_args


In [ ]:
training_args = TrainingArguments(
    output_dir="./emotion_model",
    eval_strategy="epoch",
    save_steps=99999999,
    learning_rate=3e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=4,
    weight_decay=0.01,
    warmup_steps=500,
    logging_steps=50,
)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

/tmp/ipython-input-3240721288.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Training

In [ ]:
trainer.train()

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: samiulislam1610023 (samiulislam1610023-toronto-metropolitan-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Epoch,Training Loss,Validation Loss,Accuracy
1,1.182800,1.161213,0.598193
2,1.159700,1.141313,0.598193
3,0.958100,1.185341,0.593326
4,0.787300,1.266347,0.576642


TrainOutput(global_step=6472, training_loss=1.042158078204274, metrics={'train_runtime': 640.4096, 'train_samples_per_second': 161.678, 'train_steps_per_second': 10.106, 'total_flos': 933838473048300.0, 'train_loss': 1.042158078204274, 'epoch': 4.0})

Result

In [ ]:
results = trainer.evaluate()
results

{'eval_loss': 1.2663471698760986,
 'eval_accuracy': 0.5766423357664233,
 'eval_runtime': 3.4002,
 'eval_samples_per_second': 846.116,
 'eval_steps_per_second': 52.937,
 'epoch': 4.0}

**Load Dataset**